# 02. Tool Integration

ArcAgent is the *governance* layer for tools. It registers them, signs them,
authorizes them, audits them — but it does **not** execute the agentic loop
(that's `arcrun`) or call LLMs (that's `arcllm`). This notebook walks the
tool registry surface end-to-end: how a Python function becomes a callable
tool, how the four transports (native / subprocess / MCP / HTTP) plug in,
and how `ToolError` and `ToolVetoedError` give you precise control over
what the agent can and cannot do.

You will learn:

- The arcagent ↔ arcrun split for tools (governance vs execution).
- The shape of a `RegisteredTool` and how `ToolTransport` discriminates the four wire formats.
- Building tools quickly with the `@native_tool` decorator (in-process, pure Python).
- Wiring subprocess, MCP, and HTTP transports — what code lives where.
- Catching `ToolError` (timeout, invalid args, transport failure) vs `ToolVetoedError` (pre-tool veto).
- Loading tools from a directory at runtime via `CapabilityLoader` and reading the reload diff.


## 1. Setup

Bootstrap `sys.path` so every Arc package resolves, then verify the imports we'll need.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")


The setup cell walks up from the notebook's working directory until it finds
the repo root (the directory that holds `packages/` and `pyproject.toml`),
then prepends every package's `src/` to `sys.path`. After that, the rest of
the notebook just imports — no editable installs needed.


In [ ]:
import asyncio
import json
from typing import Any
from unittest.mock import MagicMock, AsyncMock

from arcagent import ToolError, ToolVetoedError
from arcagent.core.tool_registry import (
    RegisteredTool,
    ToolRegistry,
    ToolTransport,
    native_tool,
)
from arcagent.core.config import (
    ArcAgentConfig,
    AgentConfig,
    LLMConfig,
    ToolsConfig,
    ToolConfig,
    MCPServerEntry,
    HTTPToolEntry,
    ProcessToolEntry,
)
from arcagent.core.module_bus import ModuleBus, EventContext

print("arcagent imports OK")
print("ToolTransport values:", [t.value for t in ToolTransport])


We also pull in `MagicMock`/`AsyncMock` from `unittest.mock` — the registry
expects a telemetry object that exposes `audit_event(...)` and an async
`tool_span(...)` context manager. In a real agent that's `AgentTelemetry`;
in this notebook we mock it so every cell is hermetic.


In [ ]:
def make_mock_telemetry() -> MagicMock:
    """Minimal telemetry double — audit_event() + async tool_span()."""
    tel = MagicMock()
    tel.audit_event = MagicMock()
    span = MagicMock()
    span.__aenter__ = AsyncMock(return_value=MagicMock())
    span.__aexit__ = AsyncMock(return_value=False)
    tel.tool_span = MagicMock(return_value=span)
    return tel


def make_registry(*, allow=None, deny=None) -> ToolRegistry:
    """Factory: a fresh registry with empty policy and no real telemetry."""
    config = ArcAgentConfig(
        agent=AgentConfig(name="walkthrough"),
        llm=LLMConfig(model="anthropic/claude-sonnet-4-6"),
        tools=ToolsConfig(policy=ToolConfig(allow=allow or [], deny=deny or [])),
    )
    return ToolRegistry(
        config=config.tools,
        bus=ModuleBus(),
        telemetry=make_mock_telemetry(),
        agent_did="did:arc:walkthrough",
        tier="personal",
    )


reg = make_registry()
print("registry:", reg, "tools:", list(reg.tools))


## 2. Tools in arcagent vs arcrun

The repo is strict about layering (see top-level `CLAUDE.md`):

| Concern | Owner |
|---|---|
| LLM calls (provider API, retries, fallback, audit) | `arcllm` |
| The agentic loop (turn taking, tool dispatch, events) | `arcrun` |
| Tool *governance* (register, classify, policy, audit, identity binding) | `arcagent` |

That means **arcagent never executes a tool by itself**. It builds a list of
`arcrun.Tool` objects via `ToolRegistry.to_arcrun_tools()`; arcrun's loop
calls them. Every `arcrun.Tool.execute` callable is a wrapper produced by
arcagent that injects:

1. JSON-Schema argument validation.
2. Policy pipeline check (first-DENY-wins, fail-closed) — see notebook 03.
3. `agent:pre_tool` event (allows a module to veto → `ToolVetoedError`).
4. Telemetry span + `asyncio.wait_for(...)` timeout.
5. `agent:post_tool` event.
6. Audit event with `actor_did` and `tier`.

So the *physical* call site is in arcrun. The *policy* is in arcagent. The
arrow goes one way: arcagent → arcrun (never the reverse).


In [ ]:
# Concrete demonstration: register a tool, then look at what arcrun receives.
async def demo_layering():
    registry = make_registry()

    @native_tool(description="Add two integers", source="math")
    async def add_tool(a: int = 0, b: int = 0) -> int:
        return a + b

    registry.register(add_tool.tool)
    arcrun_tools = registry.to_arcrun_tools()
    return registry, arcrun_tools


registry, arcrun_tools = await (demo_layering())
print("arcagent registered:", list(registry.tools))
print("arcrun receives:", [t.name for t in arcrun_tools])
print("arcrun tool execute is wrapped:", callable(arcrun_tools[0].execute))
print("arcrun tool timeout_seconds:", arcrun_tools[0].timeout_seconds, "(arcagent owns timeouts)")


Note that `arcrun_tools[0].timeout_seconds is None` — arcagent's wrapper
handles the timeout via `asyncio.wait_for(...)` so arcrun does not also
race a second timer against the same call (double-timeout is the fastest
way to corrupt a span and lose audit events).


## 3. CapabilityRegistry — registering a tool, ToolEntry shape, lookup

`ToolRegistry` (in `core/tool_registry.py`) is the *agent-facing* registry
used at runtime. Sitting underneath it is `CapabilityRegistry` (in
`core/capability_registry.py`) — a thread-safe, kind-discriminated store
that holds `ToolEntry`, `SkillEntry`, `HookEntry`, and friends. The
`CapabilityLoader` (next section) feeds it; the prompt assembly subscriber
reads it; conflicts resolve as last-wins for tools/skills, fan-out for hooks,
drain-then-replace for background tasks.

The two registries layer cleanly:

```
ToolRegistry          ← runtime dispatch, security wrapping, arcrun bridge
   └── CapabilityRegistry  ← structural store (tools/skills/hooks/tasks/caps)
```


In [ ]:
from arcagent.core.capability_registry import (
    CapabilityRegistry,
    ToolEntry,
    SkillEntry,
    HookEntry,
    BackgroundTaskEntry,
    LifecycleEntry,
    RegisterResult,
    Kind,
)
from arcagent.tools._decorator import ToolMetadata
from pathlib import Path

print("Kind union:", Kind)
print("CapabilityRegistry exports:", [
    c.__name__
    for c in (CapabilityRegistry, ToolEntry, SkillEntry, HookEntry,
              BackgroundTaskEntry, LifecycleEntry, RegisterResult)
])


A `ToolEntry` is the *frozen* record the registry stores. Its
`meta: ToolMetadata` carries the public name, version, classification
(`read_only` vs `state_modifying`), and JSON Schema; `execute` is the
async callable; `source_path` and `scan_root` record provenance — the
loader uses `scan_root in {"builtins", "global", "agent", "workspace"}`
to decide whether the tool went through the AST validator and the
trust-on-first-use policy gate.


In [ ]:
async def demo_capability_registry():
    cr = CapabilityRegistry(agent_did="did:arc:walkthrough", tier="personal")

    async def echo(text: str = "") -> str:
        return f"echo: {text}"

    meta = ToolMetadata(
        name="echo",
        version="1.0.0",
        description="Echo text back",
        classification="read_only",
        input_schema={
            "type": "object",
            "properties": {"text": {"type": "string"}},
        },
    )
    fake_path = Path("/var/echo.py")  # stand-in path; loader never touches disk here
    entry = ToolEntry(
        meta=meta,
        execute=echo,
        source_path=fake_path,
        scan_root="builtins",
    )

    result = await cr.register_tool(entry)
    print("register outcome:", result.outcome)

    looked_up = await cr.get_tool("echo")
    assert looked_up is entry
    print("looked up:", looked_up.meta.name, "from", looked_up.scan_root)

    # Re-register the same name with a bumped version → "replaced"
    meta2 = ToolMetadata(
        name="echo", version="1.1.0", description="Echo (improved)",
        classification="read_only",
        input_schema=meta.input_schema,
    )
    entry2 = ToolEntry(meta=meta2, execute=echo,
                      source_path=fake_path, scan_root="agent")
    result2 = await cr.register_tool(entry2)
    print("re-register outcome:", result2.outcome,
          "previous_version:", result2.previous_version)


await (demo_capability_registry())


The `RegisterResult` is what the loader uses to render the human-readable
reload diff (`reload: +1 added, ~1 replaced (echo 1.0.0→1.1.0), -0 removed`).
Last-wins semantics are deliberate — they let an agent override a built-in
echo tool with a per-agent variant just by dropping a Python file under
`<agent_root>/capabilities/`.


## 4. In-process (`NATIVE`) transport

The simplest tool is a pure async Python function. The `@native_tool`
decorator does three things:

1. Builds a JSON Schema from the function signature and a `params=` hint dict.
2. Wraps the function in a `RegisteredTool` with `transport=ToolTransport.NATIVE`.
3. Stamps the original function with a `.tool` attribute so you can retrieve it.

Native tools run in the same event loop as the agent, share its memory, and
have zero serialization overhead. They are the fastest transport but also
the *least isolated* — a native tool can see anything the agent can see.
For untrusted code, use subprocess or MCP.


In [ ]:
@native_tool(
    description="Read a key from an in-memory dict",
    source="walkthrough",
    params={"key": "Dictionary key to read"},
    required=["key"],
    when_to_use="When you need to look up a value by key",
    example="kv_get(key='alpha')",
    category="storage",
)
async def kv_get(key: str = "") -> str:
    store = {"alpha": "1", "beta": "2", "gamma": "3"}
    if key not in store:
        raise KeyError(key)
    return store[key]


print("decorated function name:", kv_get.__name__)
print("schema:", json.dumps(kv_get.tool.input_schema, indent=2))
print("transport:", kv_get.tool.transport)


The decorator inferred `type: string` for `key` from the annotation and
copied the description from the `params=` dict. Now register the tool and
exercise the wrapped execute path — that's where the audit event and
`agent:pre_tool` event fire.


In [ ]:
async def run_native_tool():
    registry = make_registry()
    registry.register(kv_get.tool)
    wrapped = registry._create_wrapped_execute(kv_get.tool)
    result = await wrapped({"key": "alpha"})
    return result, registry


result, registry_after = await (run_native_tool())
print("result:", result)
print("registered:", list(registry_after.tools))

# The audit_event mock was called — confirm tool.executed fired.
calls = registry_after._telemetry.audit_event.call_args_list
print("audit events emitted:", [c.args[0] for c in calls])


The tool returned `"1"` and a `tool.executed` audit event was recorded.
The registry's wrapper also invoked the schema validator before calling
`kv_get` — try a missing required arg to see what the error looks like.


In [ ]:
async def missing_required():
    registry = make_registry()
    registry.register(kv_get.tool)
    wrapped = registry._create_wrapped_execute(kv_get.tool)
    try:
        await wrapped({})  # missing required 'key'
    except ToolError as exc:
        return exc
    return None


err = await (missing_required())
print("error code:", err.code)
print("error message:", err.message)
print("error component:", err.component)


## 5. Subprocess (`PROCESS`) transport

Subprocess tools run in a child process. Use them when you need:

- **Resource isolation** — separate memory space, separate file descriptors,
  ability to ulimit / cgroup the child.
- **Crash containment** — a SIGSEGV in a subprocess does not take the agent down.
- **Untrusted code** — the federal tier escalates to Firecracker microVMs
  for fully untrusted execution; the `PROCESS` transport is the soft variant.

`ToolsConfig.process` is a dict of `ProcessToolEntry` (command + args + timeout).
At runtime arcagent constructs a `RegisteredTool` whose `execute` callable
shells out via `asyncio.create_subprocess_exec(...)`, captures stdout, and
returns it as the tool result.


In [ ]:
print("ProcessToolEntry fields:", ProcessToolEntry.model_fields.keys())

example_entry = ProcessToolEntry(
    command="python3",
    args=["-c", "import json,sys; print(json.dumps({'msg':'hello from child'}))"],
    timeout_seconds=10,
)
print("example process entry:", example_entry.model_dump())


Below we wire that entry into a `RegisteredTool` manually so you can see
the moving parts. In production the loader does this for you when the
agent's `arcagent.toml` declares `[tools.process.<name>]` — but you can
also register transport-specific tools by hand if you want to compose
things at runtime.


In [ ]:
async def make_subprocess_tool(entry: ProcessToolEntry, name: str) -> RegisteredTool:
    """Build a RegisteredTool that exec's a child process and returns stdout."""

    async def execute(**_kwargs: Any) -> str:
        proc = await asyncio.create_subprocess_exec(
            entry.command,
            *entry.args,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
        )
        try:
            stdout, stderr = await asyncio.wait_for(
                proc.communicate(), timeout=entry.timeout_seconds
            )
        except TimeoutError:
            proc.kill()
            await proc.wait()
            raise
        if proc.returncode != 0:
            raise ToolError(
                code="TOOL_PROCESS_FAILED",
                message=f"subprocess returned {proc.returncode}: {stderr.decode().strip()}",
                details={"returncode": proc.returncode},
            )
        return stdout.decode()

    return RegisteredTool(
        name=name,
        description="Run an isolated subprocess",
        input_schema={"type": "object", "properties": {}},
        transport=ToolTransport.PROCESS,
        execute=execute,
        timeout_seconds=entry.timeout_seconds,
        source="walkthrough",
        category="isolated",
    )


async def run_subprocess_demo():
    entry = ProcessToolEntry(
        command="python3",
        args=["-c", "import json,os; print(json.dumps({'pid': os.getpid()}))"],
        timeout_seconds=10,
    )
    tool = await make_subprocess_tool(entry, "child_pid")
    registry = make_registry()
    registry.register(tool)
    wrapped = registry._create_wrapped_execute(tool)
    raw = await wrapped({})
    return json.loads(raw)


payload = await (run_subprocess_demo())
print("child pid:", payload["pid"], "(notebook pid:", os.getpid(), ")")
print("transport used:", ToolTransport.PROCESS.value)


The child PID is different from the notebook's PID — proof the work
happened in a separate process. For federal-tier deployments, the same
contract scales up to Firecracker microVMs (per `CLAUDE.md` ASI05): the
agent never executes LLM-generated code without an OS-level isolation
boundary. The `PROCESS` transport is the personal-tier baseline.


## 6. MCP transport

[Model Context Protocol](https://modelcontextprotocol.io/) servers expose
tool catalogs over stdio (and now SSE/HTTP). `ToolsConfig.mcp_servers` is a
dict of `MCPServerEntry` describing how to launch each server; arcagent's
loader connects, lists tools, and turns each remote tool into a
`RegisteredTool` with `transport=ToolTransport.MCP`.

The notebook does **not** spawn a real MCP server (that would require a
running stdio process). Instead we build an in-memory stub that mimics the
shape of an MCP client — `list_tools()` and `call_tool(name, arguments)` —
and show how it lifts into the registry.


In [ ]:
class StubMCPClient:
    """Tiny stand-in for arcagent's MCP client wrapper.

    Real MCP clients speak JSON-RPC over stdio. The shape that matters for
    the registry is just two methods: list the tools, call one of them.
    """

    def __init__(self) -> None:
        self._catalog = {
            "translate": {
                "description": "Translate text via the (stub) MCP server",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "text": {"type": "string"},
                        "to": {"type": "string"},
                    },
                    "required": ["text", "to"],
                },
            }
        }

    async def list_tools(self) -> list[dict[str, Any]]:
        return [{"name": n, **meta} for n, meta in self._catalog.items()]

    async def call_tool(self, name: str, arguments: dict[str, Any]) -> dict[str, Any]:
        if name not in self._catalog:
            raise KeyError(name)
        # Pretend the server translated something.
        return {
            "content": [
                {"type": "text", "text": f"[{arguments['to']}] {arguments['text']}"}
            ]
        }


print("StubMCPClient ready — mimics list_tools / call_tool JSON-RPC shape")


Now lift each server-side tool into a `RegisteredTool`. The execute
callable closes over the client and the remote tool name. Note how the
registry doesn't care that the bytes leave the process — once the tool is
registered, the wrapper applies policy, audit, and timeout exactly as for
a native tool. **That uniformity is the point.**


In [ ]:
async def attach_mcp_server(server_name: str, client: StubMCPClient) -> list[RegisteredTool]:
    """Translate an MCP server's catalog into RegisteredTool list."""
    tools: list[RegisteredTool] = []
    for entry in await client.list_tools():
        remote_name = entry["name"]

        async def execute(_remote=remote_name, _client=client, **kwargs: Any) -> str:
            payload = await _client.call_tool(_remote, kwargs)
            # MCP responses use a content-block list; extract the first text block.
            for block in payload.get("content", []):
                if block.get("type") == "text":
                    return block["text"]
            return json.dumps(payload)

        tools.append(
            RegisteredTool(
                name=f"{server_name}.{remote_name}",
                description=entry["description"],
                input_schema=entry["input_schema"],
                transport=ToolTransport.MCP,
                execute=execute,
                timeout_seconds=30,
                source=f"mcp:{server_name}",
                category="external",
            )
        )
    return tools


async def run_mcp_demo():
    registry = make_registry()
    client = StubMCPClient()
    for tool in await attach_mcp_server("translate-server", client):
        registry.register(tool)
    wrapped = registry._create_wrapped_execute(
        registry.tools["translate-server.translate"]
    )
    return await wrapped({"text": "hello", "to": "fr"})


print("MCP stub result:", await (run_mcp_demo()))


The `MCPServerEntry` config type tells arcagent how to *launch* the real
server (command, args, env). The notebook's stub skips the launch step —
in production, the loader spawns the server, opens a stdio pipe, performs
the MCP handshake, then runs `list_tools()` exactly as we did above.


In [ ]:
print("MCPServerEntry fields:", list(MCPServerEntry.model_fields))
example_mcp = MCPServerEntry(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", "/var/data"],
    env={"NODE_NO_WARNINGS": "1"},
    timeout_seconds=30,
)
print("example mcp entry:", example_mcp.model_dump())


## 7. HTTP transport

`HTTPToolEntry` describes a tool that calls a remote HTTP endpoint —
typically a tools-as-a-service vendor or an internal microservice. The
agent serializes arguments as JSON, POSTs (or GETs) to the URL, and
returns the response body.

We mock `httpx` with a fake transport so the cell is hermetic. Real
deployments would use `httpx.AsyncClient` with mTLS for federal tier.


In [ ]:
print("HTTPToolEntry fields:", list(HTTPToolEntry.model_fields))


class FakeHTTPClient:
    """Pretends to be httpx.AsyncClient for one request."""

    def __init__(self, response_payload: dict[str, Any]) -> None:
        self._payload = response_payload
        self.calls: list[dict[str, Any]] = []

    async def post(
        self, url: str, *, json: dict[str, Any], headers: dict[str, str]
    ) -> "FakeResponse":
        self.calls.append({"url": url, "json": json, "headers": headers})
        return FakeResponse(self._payload)


class FakeResponse:
    def __init__(self, body: dict[str, Any]) -> None:
        self._body = body
        self.status_code = 200

    def json(self) -> dict[str, Any]:
        return self._body

    def raise_for_status(self) -> None:
        return None


client = FakeHTTPClient({"summary": "all systems nominal"})
print("fake http client ready")


Wire the entry into a `RegisteredTool`. Note how transport-specific
errors (HTTP 5xx, timeouts, bad JSON) get translated into `ToolError`
with a stable `code` so callers can branch on the error class without
caring whether it came from native, MCP, subprocess, or HTTP.


In [ ]:
def make_http_tool(entry: HTTPToolEntry, name: str, http_client: FakeHTTPClient) -> RegisteredTool:
    async def execute(**kwargs: Any) -> str:
        try:
            response = await http_client.post(
                entry.url, json=kwargs, headers=entry.headers
            )
            response.raise_for_status()
            return json.dumps(response.json())
        except Exception as exc:
            raise ToolError(
                code="TOOL_HTTP_FAILED",
                message=f"HTTP call to {entry.url} failed: {exc}",
                details={"url": entry.url},
            ) from exc

    return RegisteredTool(
        name=name,
        description="Summarize via remote HTTP service",
        input_schema={
            "type": "object",
            "properties": {"text": {"type": "string"}},
            "required": ["text"],
        },
        transport=ToolTransport.HTTP,
        execute=execute,
        timeout_seconds=entry.timeout_seconds,
        source="walkthrough-http",
        category="external",
    )


async def run_http_demo():
    entry = HTTPToolEntry(
        url="https://example.invalid/summarize",
        method="POST",
        headers={"X-Trace-Id": "wt-02"},
        timeout_seconds=15,
    )
    tool = make_http_tool(entry, "remote_summarize", client)
    registry = make_registry()
    registry.register(tool)
    wrapped = registry._create_wrapped_execute(tool)
    return await wrapped({"text": "lorem ipsum"})


print("http result:", await (run_http_demo()))
print("recorded HTTP calls:", client.calls)


Same wrapper, same audit event, same `ToolError` taxonomy. A module
listening to `agent:post_tool` cannot tell which transport produced the
result without inspecting the tool's `transport` attribute — and that's
the design goal. Transports are an implementation detail; the *contract*
is a single uniform tool-call surface.


## 8. `ToolError` vs `ToolVetoedError`

The two errors have completely different meanings:

| Error | Meaning | Typical cause |
|---|---|---|
| `ToolError` | The tool tried to run but failed | Timeout, transport failure, invalid args, exception inside the tool |
| `ToolVetoedError` | The tool **never ran** because a pre-tool handler said no | Human-in-the-loop block, tier policy, capability composition check |

`ToolVetoedError` extends `ToolError`, so a single `except ToolError`
catches both — but if you need to distinguish (e.g. to retry on transient
failures but never on a veto), catch `ToolVetoedError` first.


In [ ]:
print("ToolError MRO:", [c.__name__ for c in ToolError.__mro__[:4]])
print("ToolVetoedError MRO:", [c.__name__ for c in ToolVetoedError.__mro__[:5]])
print("issubclass(ToolVetoedError, ToolError):", issubclass(ToolVetoedError, ToolError))


**Triggering a `ToolError`** — the easiest way is a tool that raises.
The wrapper catches the timeout case explicitly; for other exceptions the
underlying error propagates wrapped in the audit trail.


In [ ]:
async def raise_a_tool_error():
    @native_tool(description="Always fails")
    async def boom() -> str:
        raise RuntimeError("intentional failure")

    registry = make_registry()
    registry.register(boom.tool)
    wrapped = registry._create_wrapped_execute(boom.tool)
    try:
        await wrapped({})
    except (ToolError, RuntimeError) as exc:
        return exc


err = await (raise_a_tool_error())
print("type:", type(err).__name__, "—", err)


**Triggering a timeout** — register a tool that sleeps longer than its
`timeout_seconds` and watch for `code='TOOL_TIMEOUT'`.


In [ ]:
async def trigger_timeout():
    async def slow(**_kwargs: Any) -> str:
        await asyncio.sleep(2)
        return "never"

    tool = RegisteredTool(
        name="slow",
        description="too slow",
        input_schema={"type": "object", "properties": {}},
        transport=ToolTransport.NATIVE,
        execute=slow,
        timeout_seconds=1,
    )
    registry = make_registry()
    registry.register(tool)
    wrapped = registry._create_wrapped_execute(tool)
    try:
        await wrapped({})
    except ToolError as exc:
        return exc


err = await (trigger_timeout())
print("code:", err.code)
print("details:", err.details)


**Triggering a `ToolVetoedError`** — subscribe a handler to
`agent:pre_tool` that calls `ctx.veto(reason)`. The wrapper inspects
`ctx.is_vetoed` after emitting the event and short-circuits with
`ToolVetoedError`. Audit and post_tool events are *not* emitted on a veto
because the tool never ran.


In [ ]:
async def trigger_veto():
    bus = ModuleBus()
    config = ArcAgentConfig(
        agent=AgentConfig(name="walkthrough"),
        llm=LLMConfig(model="anthropic/claude-sonnet-4-6"),
    )
    registry = ToolRegistry(
        config=config.tools, bus=bus, telemetry=make_mock_telemetry(),
        agent_did="did:arc:walkthrough", tier="personal",
    )

    async def block_dangerous(ctx: EventContext) -> None:
        if ctx.data["tool"] == "danger":
            ctx.veto("blocked by walkthrough policy module")

    bus.subscribe("agent:pre_tool", block_dangerous, priority=10)

    @native_tool(description="dangerous", source="demo")
    async def danger() -> str:
        return "should never run"

    registry.register(danger.tool)
    wrapped = registry._create_wrapped_execute(danger.tool)
    try:
        await wrapped({})
    except ToolVetoedError as exc:
        return exc


err = await (trigger_veto())
print("type:", type(err).__name__)
print("code:", err.code, "(VETO is its own stable error code)")
print("message:", err.message)
print("details:", err.details)


The veto reason is preserved in `details["reason"]` so the LLM can read
it back through the loop and decide whether to ask the user, try a
different tool, or give up. That feedback path is what keeps a federal
tier deployment usable — denials are *informative*, not silent.


## 9. `CapabilityLoader` — picking up tools from a directory

In a real agent you don't construct `ToolEntry` instances by hand. You drop
a `.py` file in one of four scan roots:

| Precedence | Root | Trust |
|---|---|---|
| 1 (highest) | `arcagent/builtins/capabilities/` | trusted (ships with the package) |
| 2 | `~/.arc/capabilities/` | trusted (operator-curated) |
| 3 | `<agent_root>/capabilities/` | trusted (per-agent) |
| 4 | `<agent_root>/workspace/.capabilities/` | **untrusted** — AST-validated, sandboxed |

Each `.py` file declares functions with the `@tool` decorator (or
`@hook`, `@background_task`, `@capability` for other kinds). The loader
imports each file once, finds the decorated values, and registers them.

`reload()` returns a one-line diff: `+N added, ~M replaced, -K removed,
0 errors`. Errors render on indented sub-lines.


In [ ]:
import tempfile
from arcagent.core.capability_loader import CapabilityLoader, ScanRoot


async def demo_loader():
    cr = CapabilityRegistry(agent_did="did:arc:walkthrough", tier="personal")

    with tempfile.TemporaryDirectory() as tmpdir:
        scan_dir = Path(tmpdir) / "capabilities"
        scan_dir.mkdir()

        # Drop a tool file in the scan root.
        tool_src = '''
from arcagent.tools._decorator import tool


@tool(description="Greet someone", classification="read_only")
async def greet(name: str = "world") -> str:
    return f"hello, {name}"
'''
        (scan_dir / "greet.py").write_text(tool_src)

        loader = CapabilityLoader(
            scan_roots=[ScanRoot(("agent", scan_dir))],
            registry=cr,
        )

        # First reload — should report "+1 added (greet)"
        diff1 = await loader.reload()
        # Second reload, no changes — "+0 added, ~0 replaced, -0 removed"
        diff2 = await loader.reload()

        # Bump the file's version → "~1 replaced (greet 1.0.0→1.1.0)"
        bumped = tool_src.replace(
            '@tool(description="Greet someone", classification="read_only")',
            '@tool(description="Greet someone", version="1.1.0", classification="read_only")',
        )
        (scan_dir / "greet.py").write_text(bumped)
        diff3 = await loader.reload()

        # Remove the file → "-1 removed (greet)"
        (scan_dir / "greet.py").unlink()
        diff4 = await loader.reload()

    return diff1, diff2, diff3, diff4


diffs = await (demo_loader())
labels = ["first scan", "no-op rescan", "version bump", "removal"]
for label, diff in zip(labels, diffs, strict=True):
    print(f"{label:>14}  →  {diff}")


The four diffs walk you through the lifecycle a user actually sees in
the CLI when running `arc agent reload`. Replace and remove are detected
from the loader's *own* memory of what was registered last pass — the
registry stays the source of truth, but the loader knows what it owns.


**Untrusted scan root behavior.** When the scan root name is `"workspace"`
the loader runs every Python file through the `AstValidationCache` first
— rejecting nine categories of injection / RCE patterns (CVE-2023-37271
generator-frame bypass, CVE-2025-68668 ctypes FFI bypass, etc.). Trusted
roots skip the AST step because their authors are accountable. The list
of untrusted roots is hard-coded in `capability_loader._UNTRUSTED_ROOTS`.


In [ ]:
from arcagent.core.capability_loader import _UNTRUSTED_ROOTS

print("Untrusted scan roots:", set(_UNTRUSTED_ROOTS))
print("Trusted scan roots: builtins, global, agent")


## 10. Summary

You walked the full tool surface of arcagent:

- **Layering**: `arcagent` *governs* tools (register, classify, policy, audit, identity bind). `arcrun` *executes* them. The boundary is `ToolRegistry.to_arcrun_tools()`.
- **CapabilityRegistry**: kind-discriminated, thread-safe store for `ToolEntry`, `SkillEntry`, `HookEntry`, `BackgroundTaskEntry`, `LifecycleEntry`. Last-wins for tools, fan-out for hooks, drain-then-replace for tasks.
- **ToolRegistry**: agent-facing wrapper. Every dispatch flows through `_create_wrapped_execute` — schema validation → policy pipeline → pre_tool veto → telemetry span → execute → post_tool → audit.
- **Four transports** (`ToolTransport.NATIVE / PROCESS / MCP / HTTP`):
  - `NATIVE` via `@native_tool` — same event loop, fastest, least isolated.
  - `PROCESS` via `ProcessToolEntry` — separate PID, crash-contained.
  - `MCP` via `MCPServerEntry` — stdio JSON-RPC; we stubbed the client.
  - `HTTP` via `HTTPToolEntry` — JSON over the wire to a remote service.
- **Errors**:
  - `ToolError` — the tool ran (or tried to) and failed. Stable codes: `TOOL_INVALID_ARGS`, `TOOL_TIMEOUT`, `TOOL_HTTP_FAILED`, `TOOL_PROCESS_FAILED`.
  - `ToolVetoedError` — extends `ToolError`; the tool never ran because a pre-tool handler vetoed.
- **CapabilityLoader**: directory-driven registration. Four scan roots in precedence order; AST-validated when the root is `workspace`. `reload()` returns a one-line human-readable diff.

**Public API exercised** (from `arcagent.__init__` and `arcagent.core.*`):

- `arcagent.ToolError`, `arcagent.ToolVetoedError`
- `arcagent.core.tool_registry.ToolRegistry`, `RegisteredTool`, `ToolTransport`, `native_tool`
- `arcagent.core.capability_registry.CapabilityRegistry`, `ToolEntry`, `SkillEntry`, `HookEntry`, `BackgroundTaskEntry`, `LifecycleEntry`, `RegisterResult`, `Kind`
- `arcagent.core.capability_loader.CapabilityLoader`, `ScanRoot`
- `arcagent.core.config.ToolsConfig`, `ToolConfig`, `MCPServerEntry`, `HTTPToolEntry`, `ProcessToolEntry`
- `arcagent.core.module_bus.ModuleBus`, `EventContext`

**Next**: see `walkthroughs/arcagent/03-policy-and-modules.ipynb` for the
policy pipeline that decides *whether* a tool call is allowed, and the
module bus extension surface that makes it pluggable.
